# `ab-testing-toolkit` — scratch testing notebook

A running sanity-check sandbox. We add a section here every time a new chunk of the
library is built, so we can eyeball that things work end-to-end (separate from the
automated `pytest` suite and the polished `worked_example.ipynb`).

Run top-to-bottom; every section should execute without error.

## Scaffold smoke test

Confirm the package and its modules import.

In [1]:
import numpy as np

import abtesting
from abtesting import experiment, power, corrections, sequential, visualizations, utils

print("abtesting version:", abtesting.__version__)
print("modules imported:", [m.__name__.split('.')[-1] for m in (experiment, power, corrections, sequential, visualizations, utils)])

abtesting version: 0.1.0
modules imported: ['experiment', 'power', 'corrections', 'sequential', 'visualizations', 'utils']


## `utils`: pre-experiment checks & data cleaning

Sample-ratio-mismatch (SRM) is the *first* thing to check on any experiment: if the
realized split doesn't match the intended one, the randomization/logging is suspect
and the results aren't trustworthy.

In [2]:
from abtesting.utils import check_sample_ratio_mismatch, winsorize, log_transform, cohens_d

healthy = check_sample_ratio_mismatch(10_000, 9_950)
broken = check_sample_ratio_mismatch(10_000, 8_500)
print(f"Healthy split: mismatch={healthy.is_mismatch}  p={healthy.p_value:.3f}")
print(f"Broken split:  mismatch={broken.is_mismatch}  p={broken.p_value:.3e}")

Healthy split: mismatch=False  p=0.723
Broken split:  mismatch=True  p=2.793e-28


In [3]:
# Heavy-tailed metric (e.g. watch time): a few whales blow up the mean.
rng = np.random.default_rng(7)
watch_time = rng.lognormal(mean=2.0, sigma=1.2, size=5_000)

clipped = winsorize(watch_time, lower=0.01, upper=0.99)
logged = log_transform(watch_time)

print(f"raw mean:        {watch_time.mean():8.2f}   max: {watch_time.max():8.2f}")
print(f"winsorized mean: {clipped.mean():8.2f}   max: {clipped.max():8.2f}")
print(f"log1p mean:      {logged.mean():8.2f}   (skew tamed)")

raw mean:           14.77   max:   966.74
winsorized mean:    14.12   max:   118.04
log1p mean:          2.19   (skew tamed)


In [4]:
# Cohen's d: standardized effect size between two arms.
control = rng.normal(0.0, 1.0, 4_000)
treatment = rng.normal(0.3, 1.0, 4_000)
print(f"Cohen's d (true 0.3): {cohens_d(control, treatment):.3f}")

Cohen's d (true 0.3): 0.332
